TASK 4: Adaptive and Kalman Filtering

In [ ]:
from datetime import datetime
import time

import neurokit2 as nk
import numpy as np
import matplotlib.pyplot as plt
import scipy
from matplotlib.ticker import MaxNLocator
from datetime import timedelta

# Import Pandas only for reading Watch CSV File and for plotting description in Q1
import pandas as pd


def load_bp_csv(filename, start_datetime):
    test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')
    test = test.drop(0)
    test = test.drop(test.columns[3], axis=1)  # Drop the first column
    test = test.rename(columns={'CH9': 'ecg', 'CH1': 'ppg'})
    test['ppg'] = pd.to_numeric(test['ppg'], errors='coerce')
    # if 'min' in test.columns, create new milli sec column
    if 'min' in test.columns:
        # Convert 'min' to milliseconds
        test['time_ms'] = test['min'] * 60 * 1000
    else:
        test = test.rename(columns={'milliSec': 'time_ms'})
    test['ecg'] = pd.to_numeric(test['ecg'], errors='coerce')
    test['timestamp'] = pd.to_datetime(start_datetime) + pd.to_timedelta(test['time_ms'], unit='ms')
    return test


def load_watch_ecg(filename):
    df = pd.read_csv(filename, skiprows=2)
    df = df[['Timestamp', 'ECG data']].dropna()
    df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['ecg'] = pd.to_numeric(df['ecg'], errors='coerce')
    df = df.dropna()

    # Convert to seconds
    df['time_ms'] = (df['timestamp'] - df['timestamp'].iloc[0])
    return df


def load_watch_data(ecg_file, ppg_file):
    ecg_df = pd.read_csv(ecg_file, skiprows=2)
    ppg_df = pd.read_csv(ppg_file, skiprows=2)

    # Rename columns for clarity
    ecg_df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    ppg_df.rename(columns={'PPG Timestamp': 'timestamp', 'PPG data': 'ppg'}, inplace=True)

    latest_start_time = max(ecg_df['timestamp'].iloc[0], ppg_df['timestamp'].iloc[0])
    earliest_end_time = min(ecg_df['timestamp'].iloc[-1], ppg_df['timestamp'].iloc[-1])

    # Filter data to the common time range
    ecg_df = ecg_df[(ecg_df['timestamp'] >= latest_start_time) & (ecg_df['timestamp'] <= earliest_end_time)]
    ppg_df = ppg_df[(ppg_df['timestamp'] >= latest_start_time) & (ppg_df['timestamp'] <= earliest_end_time)]

    # Drop all columns except timestamp, time_ms and ecg/ppg
    ppg_df = ppg_df.drop(columns=['ADXL Timestamp'])
    ecg_df = ecg_df.drop(columns=['Seq No.'])
    # Merge ECG and PPG based on the millisecond given
    merged_df = pd.merge_asof(ecg_df.sort_values('timestamp'), ppg_df.sort_values('timestamp'), on='timestamp',
                              direction='nearest')

    merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
    return merged_df


def plot_watch_bp_ecg(watch_df, bp_data, start_time, end_time, dataset_name='Recording', vlines=[], vlines_name=[]):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    watch_df.sort_values('timestamp', inplace=True)
    bp_data.sort_values('time_ms', inplace=True)

    # Select time frame to plot
    # Discard time frames that are not represented in both recordings
    latest_start_time = max(watch_df['timestamp'].iloc[0], bp_data['timestamp'].iloc[0])  # + timedelta(minutes=5)
    earliest_end_time = min(watch_df['timestamp'].iloc[-1], bp_data['timestamp'].iloc[-1])
    watch_df = watch_df[(watch_df['timestamp'] >= latest_start_time) & (watch_df['timestamp'] <= earliest_end_time)]
    watch_df = watch_df[(watch_df['timestamp'] >= start_time) & (watch_df['timestamp'] <= end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= latest_start_time) & (bp_data['timestamp'] <= earliest_end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= start_time) & (bp_data['timestamp'] <= end_time)]
    watch_df = watch_df.dropna()
    # Plot ECG from Watch
    ax1.plot(watch_df['timestamp'], watch_df['PPG'], label='PPG from Watch', color='blue')
    ax1.set_ylabel('ECG Amplitude')
    ax1.set_xlabel('Time')
    ax1.set_title(f'{dataset_name} - PPG Signal from Smartwatch')

    # Plot ECG from BioPac
    ax2.plot(bp_data['timestamp'], bp_data['ecg'], label='ECG from BioPac', color='orange')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ECG Amplitude')
    ax2.set_title(f'{dataset_name} - ECG Signal from BioPac')
    ax1.legend()
    ax2.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax1.axvline(x=vline, color='red', linestyle='--', label=vline_name)
        ax2.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax1.legend()
    #ax2.legend()
    plt.tight_layout()
    plt.show()
    bp_data['ppg']
    fig, ax3 = plt.subplots(figsize=(12, 4))
    ax3.plot(bp_data['timestamp'], bp_data['ppg'], label='PPG from BioPac', color='orange')
    ax3.set_xlabel('Time')
    ax3.set_ylabel('ECG Amplitude')
    ax3.set_title(f'{dataset_name} - PPG Signal from BioPac')
    ax3.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax3.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax3.legend()
    ax3.yaxis.set_major_locator(MaxNLocator(nbins=5))
    plt.tight_layout()
    plt.show()
    # plot xyz in one plot
    fig, ax4 = plt.subplots(figsize=(12, 4))
    ax4.plot(watch_df['timestamp'], watch_df['X'], label='X-axis', color='red')
    ax4.plot(watch_df['timestamp'], watch_df['Y'], label='Y-axis', color='green')
    ax4.plot(watch_df['timestamp'], watch_df['Z'], label='Z-axis', color='purple')
    ax4.set_xlabel('Time')
    ax4.set_ylabel('Acceleration')
    ax4.set_title(f'{dataset_name} - Acceleration Data from Watch')
    ax4.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax4.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax4.legend()

    plt.tight_layout()
    plt.show()


def crop_window(data, start_time, end_time):
    data = data[(data['timestamp'] >= start_time) & (data['timestamp'] <= end_time)]
    return data


def compute_heart_rate_metrics(ecg_df, sampling_rate=1000):
    # Schritt 1: R-Peak-Detection mit neurokit2
    ecg_signal = ecg_df['ecg'].ffill().bfill()
    ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate)
    _, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)
    rpeaks_indices = rpeaks['ECG_R_Peaks']

    # Schritt 2: Instantane Herzfrequenz (ohne Toolbox)
    rr_intervals = np.diff(ecg_df['timestamp'].values[rpeaks_indices]) / np.timedelta64(1, 's')
    inst_hr = 60 / rr_intervals
    inst_hr_time = ecg_df['timestamp'].values[rpeaks_indices][1:]

    inst_hr_df = pd.DataFrame({'timestamp': inst_hr_time, 'hr': inst_hr})

    # Schritt 3: Geglättete HR (Median über 10s mit 1s Schrittweite)
    window_size = 10
    step_size = 1
    start_time = inst_hr_df['timestamp'].min()
    end_time = inst_hr_df['timestamp'].max()

    smoothed_hr = []
    times = []

    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)

        hr_vals = inst_hr_df[(inst_hr_df['timestamp'] >= current) &
                             (inst_hr_df['timestamp'] < window_end)]['hr']
        if len(hr_vals) > 0:
            smoothed_hr.append(np.median(hr_vals))
            times.append(middle)
        current += timedelta(seconds=step_size)

    smoothed_hr_df = pd.DataFrame({'timestamp': times, 'smoothed_hr': smoothed_hr})

    return inst_hr_df, smoothed_hr_df, ecg_df, rpeaks_indices


def compute_ppg_hr_metrics(ppg_df, sampling_rate=1000):
    import neurokit2 as nk

    if 'ppg' in ppg_df.columns:
        ppg_df.rename(columns={'ppg': 'PPG'}, inplace=True)

    ppg_signal = ppg_df['PPG'].ffill().bfill()
    ppg_clean = nk.ppg_clean(ppg_signal, sampling_rate=sampling_rate)
    _, peaks = nk.ppg_peaks(ppg_clean, sampling_rate=sampling_rate)
    peak_indices = peaks['PPG_Peaks']

    # Instantane Herzfrequenz
    timestamps = ppg_df['timestamp'].values
    rr_intervals = np.diff(timestamps[peak_indices]) / np.timedelta64(1, 's')
    hr = 60 / rr_intervals
    hr_time = timestamps[peak_indices][1:]

    inst_hr_df = pd.DataFrame({'timestamp': hr_time, 'hr': hr})

    # Smoothed HR: Mean in 10s window, 1s slide
    window_size = 10
    step_size = 1
    start_time = inst_hr_df['timestamp'].min() + timedelta(seconds=5)
    end_time = inst_hr_df['timestamp'].max()

    smoothed_hr, times = [], []
    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)
        hr_vals = inst_hr_df[(inst_hr_df['timestamp'] >= current) &
                             (inst_hr_df['timestamp'] < window_end)]['hr']
        if len(hr_vals) > 0:
            smoothed_hr.append(np.mean(hr_vals))
            times.append(middle)
        current += timedelta(seconds=step_size)

    smoothed_df = pd.DataFrame({'timestamp': times, 'smoothed_hr': smoothed_hr})
    return inst_hr_df, smoothed_df, ppg_df, peak_indices


watch_stairs = load_watch_data('data/assignment_3/Stairs/ecg_2025_06_12_12_00_41ecgcentrifuge.csv',
                               'data/assignment_3/Stairs/ppg_2025_06_12_12_00_41ppgcentrifuge.csv')
#2025-06-12 12:01:19.122
bp_stairs = load_bp_csv('data/assignment_3/Stairs/Biopac.txt',
                        start_datetime=datetime(2025, 6, 12, 10, 1, 3, 334000) - timedelta(seconds=-1.65))

watch_workout = load_watch_data('data/assignment_3/Workout/ecg_2025_06_12_12_42_06ecgcentrifuge.csv',
                                'data/assignment_3/Workout/ppg_2025_06_12_12_42_06ppgcentrifuge.csv')
#2025-06-12 12:42:40.750
bp_workout = load_bp_csv('data/assignment_3/Workout/Biopac.txt',
                         start_datetime=datetime(2025, 6, 12, 10, 42, 40, 750000) - timedelta(seconds=.4))

# Crop to actual workout/stair time
watch_stairs = crop_window(watch_stairs, start_time=datetime(2025, 6, 12, 10, 7, 0, 0),
                           end_time=datetime(2025, 6, 12, 10, 17, 45, 0), )
bp_stairs = crop_window(bp_stairs, start_time=datetime(2025, 6, 12, 10, 7, 0, 0),
                        end_time=datetime(2025, 6, 12, 10, 17, 45, 0), )
watch_workout = crop_window(watch_workout, start_time=datetime(2025, 6, 12, 10, 42, 47, 0),
                            end_time=datetime(2025, 6, 12, 11, 43, 00, 0), )
bp_workout = crop_window(bp_workout, start_time=datetime(2025, 6, 12, 10, 42, 47, 0),
                         end_time=datetime(2025, 6, 12, 11, 43, 00, 0), )

In [ ]:
#Q1
def normalize(signal):
    # Min-max normalization
    signal_min = np.min(signal)
    signal_max = np.max(signal)
    normalized_signal = (signal - signal_min) / (signal_max - signal_min)

    return normalized_signal, signal_min, signal_max


workout_hr_bp, sm_workout_hr_bp, workout_ecg, workout_rpeaks_bp = compute_heart_rate_metrics(bp_workout)
stairs_hr_bp, sm_stairs_hr_bp, stairs_ecg, stairs_rpeaks_bp = compute_heart_rate_metrics(bp_stairs)
workout_hr_ppg_bp, sm_workout_hr_ppg, workout_ppg, workout_rpeaks_ppg = compute_ppg_hr_metrics(bp_workout)
stairs_hr_ppg_bp, sm_stairs_hr_ppg, stairs_ppg, stairs_rpeaks_ppg = compute_ppg_hr_metrics(bp_stairs)
workout_hr_watch, sm_workout_hr_watch, workout_ecg_watch, workout_rpeaks_watch = compute_ppg_hr_metrics(watch_workout)
stairs_hr_watch, sm_stairs_hr_watch, stairs_ecg_watch, stairs_rpeaks_watch = compute_ppg_hr_metrics(watch_stairs)

sm_workout_hr_bp['hr_smooth_normalized'], sm_workout_hr_bp_min, sm_workout_hr_bp_max = normalize(
    sm_workout_hr_bp['smoothed_hr'])
sm_stairs_hr_bp['hr_smooth_normalized'], sm_stairs_hr_bp_min, sm_stairs_hr_bp_max = normalize(
    sm_stairs_hr_bp['smoothed_hr'])
sm_workout_hr_ppg['hr_smooth_normalized'], sm_workout_hr_ppg_min, sm_workout_hr_ppg_max = normalize(
    sm_workout_hr_ppg['smoothed_hr'])
sm_stairs_hr_ppg['hr_smooth_normalized'], sm_stairs_hr_ppg_min, sm_stairs_hr_ppg_max = normalize(
    sm_stairs_hr_ppg['smoothed_hr'])
sm_workout_hr_watch['hr_smooth_normalized'], sm_workout_hr_watch_min, sm_workout_hr_watch_max = normalize(
    sm_workout_hr_watch['smoothed_hr'])
sm_stairs_hr_watch['hr_smooth_normalized'], sm_stairs_hr_watch_min, sm_stairs_hr_watch_max = normalize(
    sm_stairs_hr_watch['smoothed_hr'])


def get_smoothed_motion_signals(df):
    # Assuming df has columns 'X', 'Y', 'Z' for accelerometer data
    # Generate one single signal by averaging the three axes
    x = df['X'].ffill().bfill()
    y = df['Y'].ffill().bfill()
    z = df['Z'].ffill().bfill()

    acc_mag = np.sqrt(x ** 2 + y ** 2 + z ** 2)

    # Normalize magnitude (min-max)
    df['acc_mag_norm'] = (acc_mag - np.min(acc_mag)) / (np.max(acc_mag) - np.min(acc_mag))

    # Normalize channels
    df['x_norm'] = (x - np.min(x)) / (np.max(x) - np.min(x))
    df['y_norm'] = (y - np.min(y)) / (np.max(y) - np.min(y))
    df['z_norm'] = (z - np.min(z)) / (np.max(z) - np.min(z))

    window_size = 10
    step_size = 1
    start_time = df['timestamp'].min() + timedelta(seconds=5)
    end_time = df['timestamp'].max()

    smoothed_x, smoothed_y, smoothed_z, smoothed_combined_motion, times = [], [], [], [], []
    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)
        times.append(middle)
        current += timedelta(seconds=step_size)
        smoothed_x.append(np.mean(df[(df['timestamp'] >= current) & (df['timestamp'] < window_end)]['x_norm']))
        smoothed_y.append(np.mean(df[(df['timestamp'] >= current) & (df['timestamp'] < window_end)]['y_norm']))
        smoothed_z.append(np.mean(df[(df['timestamp'] >= current) & (df['timestamp'] < window_end)]['z_norm']))
        smoothed_combined_motion.append(
            np.mean(df[(df['timestamp'] >= current) & (df['timestamp'] < window_end)]['acc_mag_norm']))

    new_df = pd.DataFrame({
        'timestamp': times,
        'x_norm': smoothed_x,
        'y_norm': smoothed_y,
        'z_norm': smoothed_z,
        'acc_mag_norm': smoothed_combined_motion
    })

    return new_df


workout_motion = get_smoothed_motion_signals(watch_workout).iloc[2:]
stairs_motion = get_smoothed_motion_signals(watch_stairs).iloc[1:]


In [ ]:
# Q2
# Merge Motion and HR data
hr_smoothed_workout_ppg = pd.merge_asof(sm_workout_hr_ppg, workout_motion, on='timestamp', direction='nearest')
hr_smoothed_stairs_ppg = pd.merge_asof(sm_stairs_hr_ppg, stairs_motion, on='timestamp', direction='nearest')
hr_smoothed_workout_ecg = pd.merge_asof(sm_workout_hr_bp, workout_motion, on='timestamp', direction='nearest')
hr_smoothed_stairs_ecg = pd.merge_asof(sm_stairs_hr_bp, stairs_motion, on='timestamp', direction='nearest')
hr_smoothed_workout_watch = pd.merge_asof(sm_workout_hr_watch, workout_motion, on='timestamp', direction='nearest')
hr_smoothed_stairs_watch = pd.merge_asof(sm_stairs_hr_watch, stairs_motion, on='timestamp', direction='nearest')


def denormalize(normalized_signal, min_val, max_val):
    return normalized_signal * (max_val - min_val) + min_val


def lms_filter(reference_signal, system_input, num_coeffs=8, learning_rate=0.01):
    N = len(system_input)
    w = np.zeros(num_coeffs)
    output_signal = np.zeros(N)
    x_pad = np.concatenate([np.zeros(num_coeffs - 1), reference_signal])

    for n in range(N):
        x_vec = x_pad[n: n + num_coeffs][::-1]
        y = np.dot(w, x_vec)
        e = system_input[n] - y
        output_signal[n] = e
        w += 2 * learning_rate * e * x_vec

    return output_signal, w


# Filter the smoothed HR data using LMS filter
hr_smoothed_workout_ppg['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_workout_ppg['acc_mag_norm'],
    system_input=hr_smoothed_workout_ppg['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)
hr_smoothed_stairs_ppg['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_stairs_ppg['acc_mag_norm'],
    system_input=hr_smoothed_stairs_ppg['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)
hr_smoothed_workout_ecg['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_workout_ecg['acc_mag_norm'],
    system_input=hr_smoothed_workout_ecg['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)
hr_smoothed_stairs_ecg['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_stairs_ecg['acc_mag_norm'],
    system_input=hr_smoothed_stairs_ecg['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)
hr_smoothed_workout_watch['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_workout_watch['acc_mag_norm'],
    system_input=hr_smoothed_workout_watch['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)
hr_smoothed_stairs_watch['filtered_hr'], _ = lms_filter(
    reference_signal=hr_smoothed_stairs_watch['acc_mag_norm'],
    system_input=hr_smoothed_stairs_watch['hr_smooth_normalized'],
    num_coeffs=8,
    learning_rate=0.01
)

# Denormalize the filtered HR signals
hr_smoothed_workout_ppg['filtered_hr_denorm'] = denormalize(
    hr_smoothed_workout_ppg['filtered_hr'], sm_workout_hr_ppg_min, sm_workout_hr_ppg_max)
hr_smoothed_stairs_ppg['filtered_hr_denorm'] = denormalize(
    hr_smoothed_stairs_ppg['filtered_hr'], sm_stairs_hr_ppg_min, sm_stairs_hr_ppg_max)
hr_smoothed_workout_ecg['filtered_hr_denorm'] = denormalize(
    hr_smoothed_workout_ecg['filtered_hr'], sm_workout_hr_bp_min, sm_workout_hr_bp_max)
hr_smoothed_stairs_ecg['filtered_hr_denorm'] = denormalize(
    hr_smoothed_stairs_ecg['filtered_hr'], sm_stairs_hr_bp_min, sm_stairs_hr_bp_max)
hr_smoothed_workout_watch['filtered_hr_denorm'] = denormalize(
    hr_smoothed_workout_watch['filtered_hr'], sm_workout_hr_watch_min, sm_workout_hr_watch_max)
hr_smoothed_stairs_watch['filtered_hr_denorm'] = denormalize(
    hr_smoothed_stairs_watch['filtered_hr'], sm_stairs_hr_watch_min, sm_stairs_hr_watch_max)

# Plotting Two of the Signals
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hr_smoothed_workout_ecg['timestamp'], hr_smoothed_workout_ecg['smoothed_hr'], label='Smoothed HR from BP ECG', color='blue')
ax.plot(hr_smoothed_workout_ppg['timestamp'], hr_smoothed_workout_ppg['filtered_hr_denorm'], label='Filtered HR from BP PPG', color='orange')
ax.plot(hr_smoothed_workout_ppg['timestamp'], hr_smoothed_workout_ppg['smoothed_hr'], label='Unfiltered HR from BP PPG', color='red')
ax.plot(hr_smoothed_workout_watch['timestamp'], hr_smoothed_workout_watch['filtered_hr_denorm'], label='Filtered HR from SmartWatch PPG', color='green')
ax.plot(hr_smoothed_workout_watch['timestamp'], hr_smoothed_workout_watch['smoothed_hr'], label='Unfiltered HR from SmartWatch PPG', color='purple')
ax.set_xlabel('Time (Datetime) Hours:Minutes:Seconds Timezone UTC Universal Time Code +0')
ax.set_ylabel('Heart Rate in BPM Beats Per Minute 1/60HZ Hertz')
from matplotlib.dates import DateFormatter
ax.xaxis.set_major_formatter(DateFormatter('%H:%M:%S'))
ax.set_title('Workout Heart Rate Comparison Q1 - LMS Filtered vs Unfiltered - Ground Truth from Smoothed HR from BP ECG')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Q3



In [ ]:
# Q4

